# ALFRED transformer fine-tuning

Select a GPU runtime. In Google Drive, put a copy of this checkout (including the Task 03 scripts and the original `data/processed/{train,validation,test}.csv`) in `MyDrive/robot-command-edge-ai`. The notebook writes artifacts to that Drive checkout so they survive a runtime reset. Existing run directories are preserved. Use a new output directory for a fresh run.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/robot-command-edge-ai
%pip install -r requirements-transformers.txt

In [ ]:
!nvidia-smi
!python -c 'import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(), torch.cuda.is_bf16_supported())'
!python -m unittest discover -s tests -v

In [ ]:
import subprocess
import sys
from pathlib import Path

models = ("bert-base-uncased", "distilbert-base-uncased")
output = Path("artifacts/transformers-colab")
batch_args = ["--batch-size", "16", "--accumulation", "1", "--batch-reason", "Task 03 defaults"]

def run(module, *args):
    subprocess.run([sys.executable, "-m", module, *map(str, args)], check=True)

for model in models:
    run("scripts.train_transformers", "--model", model, "--require-cuda",
        "--output-dir", output, "--smoke", *batch_args)

If a smoke run hits CUDA out-of-memory, use a fresh output directory, set microbatch size to 8 and accumulation to 2 for both models, record the reason in `batch_args`, and repeat both smoke runs before full training.

In [ ]:
for model in models:
    run("scripts.train_transformers", "--model", model, "--require-cuda",
        "--output-dir", output, *batch_args)

In [ ]:
if not Path("artifacts/ml-baselines/metrics.json").is_file():
    run("scripts.train_ml_baselines")
run("scripts.evaluate_transformers", "--output-dir", output)